# Step 3: Train RF-DETR on Swimming Pools

Trains RF-DETR Nano (mandatory) and Small (optional) on the manually-cleaned Roboflow export. Uses **COCO format** (RF-DETR requirement, unlike YOLO26).

**Runtime:** Colab → Runtime → Change runtime type → **A100 GPU** (40 GB; preferred since RF-DETR is heavier than YOLO and we now use batch_size=16 with no grad accumulation).

**Dataset input:** re-export the same Roboflow version in **COCO** format and place on Drive. Same review work, only the export format differs.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU, switch runtime to GPU.'
print(f'GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB  |  torch {torch.__version__}')

In [ ]:
%pip install -q 'rfdetr[train,loggers]>=1.4.0' supervision faster-coco-eval torchmetrics
import rfdetr
from importlib.metadata import version
print('rfdetr', version('rfdetr'))

## Load COCO dataset from Drive

Expects the COCO export at `MyDrive/IE/IndividualAssignmentMBD2026/pool_coco.zip`. Re-export from Roboflow: same Version → Export → format = **COCO** → download zip → upload to Drive at that path (and name it `pool_coco.zip`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, pathlib, shutil, json

# -- CONFIG ------------------------------------------------------------------
# Same folder as Step 2. Searched recursively. Change only if your zips live elsewhere.
ROBOFLOW_DIR = '/content/drive/MyDrive/IE/IndividualAssignmentMBD2026'
# ---------------------------------------------------------------------------

def find_roboflow_zip(root, kind):
    """Locate a Roboflow export by INSPECTING CONTENTS (filename-agnostic).
    kind: 'coco' -> zip containing _annotations.coco.json. Returns (path, is_roboflow) or (None, False)."""
    root = pathlib.Path(root)
    hits = []
    for z in sorted(root.rglob('*.zip')):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
                is_rf = any('readme.roboflow' in n.lower() for n in names)
                if kind == 'coco' and any(n.endswith('_annotations.coco.json') for n in names):
                    hits.append((z, is_rf))
        except zipfile.BadZipFile:
            continue
    hits.sort(key=lambda h: (not h[1]))
    return hits[0] if hits else (None, False)

ZIP_PATH, _ = find_roboflow_zip(ROBOFLOW_DIR, 'coco')
assert ZIP_PATH is not None, (
    f'No COCO export (_annotations.coco.json) found under {ROBOFLOW_DIR}.\n'
    'Re-export the SAME Roboflow version in COCO format and upload it there (or fix ROBOFLOW_DIR).'
)
print('Using COCO export:', ZIP_PATH)

DATA_DIR = pathlib.Path('/content/pool_coco')
if DATA_DIR.exists(): shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True)
with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(DATA_DIR)

# Locate the COCO root (the dir whose children are train/valid/test), handling nested zips.
ann_files = list(DATA_DIR.rglob('_annotations.coco.json'))
assert ann_files, 'Extracted COCO export contains no _annotations.coco.json.'
COCO_ROOT = ann_files[0].parent.parent
DATA_DIR = COCO_ROOT

# Roboflow COCO layout: train/, valid/, test/ each with images + _annotations.coco.json
for split in ['train', 'valid', 'test']:
    sp = DATA_DIR / split
    if not sp.exists():
        print(f'WARNING: no {split} split')
        continue
    ann = sp / '_annotations.coco.json'
    with open(ann) as f: coco = json.load(f)
    print(f'{split}: {len(coco["images"])} images, {len(coco["annotations"])} annotations, classes: {[c["name"] for c in coco["categories"]]}')
DATASET_DIR = str(DATA_DIR)
print('Dataset root:', DATASET_DIR)

## Training configuration

| Parameter | Value | Justification |
|---|---|---|
| Backbone | DINOv2 (frozen for first epochs by default) | RF-DETR's pretrained self-supervised ViT backbone. Freezing it early protects the transferred features while the detection head warms up — same transfer-learning principle as the lectures (freeze pretrained layers, train the new head first). |
| Optimizer | AdamW | RF-DETR default; standard for transformer training. |
| Learning rate | 1e-4 (head), ~1e-5 (backbone) | DETR-family standard; the pretrained backbone gets a ~10x smaller LR than the randomly-initialised head. Used at RF-DETR defaults (we don't override them). |
| LR scheduler | Step decay (default) | RF-DETR's built-in schedule. |
| **Epochs** | **100** | **Matches YOLO26 (Step 2) so the architecture comparison isn't confounded by training budget** — this is the comparison the brief explicitly grades. RF-DETR often converges sooner (pretrained backbone + Hungarian matching); 100 just gives a fair ceiling. Set to 50 if you need to halve RF-DETR wall-clock. |
| **Image size (resolution)** | **672** | **Not 640:** RF-DETR's DINOv2 tiles images into 14x14 patches and the windowed variants need divisibility by 56 — 640 satisfies neither and will error / silently fall back. 672 is divisible by 14, 24, 32 **and** 56, so it's valid on every RF-DETR version, and it's the closest valid value to YOLO's 640 (~5% gap -> note as a minor fairness caveat). Set on the model **constructor** (RF-DETR's documented way), not just train(). |
| **Batch size** | **8** | 672 is heavier than RF-DETR's 512 default; batch 8 keeps RF-DETR Small within A100 40 GB. Bump to 16 if you have headroom at your resolution. |
| **Grad accumulation** | **2** | Effective batch = 8 x 2 = **16**, matching YOLO26's effective batch for a fair comparison. |
| Data augmentation | RF-DETR defaults | RF-DETR applies its own train-time augmentation internally (multi-scale, flips, photometric). Report this as "library defaults" — we don't add YOLO-style aug here because the pipelines differ. |
| Random seed | 0 | Reproducibility; same seed as Step 2. |

> If you hit a CUDA OOM on RF-DETR Small at 672, drop to `batch_size=4, grad_accum_steps=4` (still effective batch 16). If a constructor rejects the `resolution=` kwarg on your installed version, the helper falls back to the model's native default (Nano 384 / Small 512) and prints a note.

In [ ]:
# Resolution is set on the model CONSTRUCTOR (see make_model below), so it is NOT in TRAIN_CFG.
RESOLUTION = 672   # divisible by 14/24/32/56 -> valid for every RF-DETR variant; closest valid value to YOLO's 640

TRAIN_CFG = dict(
    dataset_dir=DATASET_DIR,
    epochs=100,            # matches YOLO26 (Step 2) for a fair, budget-controlled architecture comparison
    batch_size=8,          # 672 res is heavier than the 512 default; 8 fits RF-DETR Small on A100 40 GB
    grad_accum_steps=2,    # effective batch = batch_size x grad_accum_steps = 16 (matches YOLO26)
    seed=0,                # matches Step 2's seed; TrainConfig.seed (rfdetr config.py:705) accepts int
)
print(f'RESOLUTION = {RESOLUTION}  (set on the model constructor)')
for k, v in TRAIN_CFG.items(): print(f'  {k:18s} = {v}')

## Train RF-DETR Nano

In [ ]:
import gc, time, shutil, pathlib
import torch.nn as nn
from rfdetr import RFDETRNano, RFDETRSmall

rfdetr_results = {}

def make_model(cls, **extra):
    """Instantiate an RF-DETR model at RESOLUTION (set on the CONSTRUCTOR — RF-DETR's documented way,
    so train- and inference-time resolution always match). Falls back to the model's native default
    (Nano 384 / Small 512) if this installed version's constructor doesn't accept `resolution`."""
    try:
        return cls(resolution=RESOLUTION, **extra)
    except TypeError:
        print(f'  note: {cls.__name__} ignored resolution kwarg; using its native default resolution')
        return cls(**extra)

def _free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    print(f'  GPU mem: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated / {torch.cuda.memory_reserved()/1e9:.2f} GB reserved')

def _count_params(obj, depth=0, seen=None):
    """Walk RF-DETR wrapper objects to find the underlying nn.Module and count parameters."""
    if seen is None: seen = set()
    if depth > 4 or id(obj) in seen: return 0
    seen.add(id(obj))
    if isinstance(obj, nn.Module):
        return sum(p.numel() for p in obj.parameters())
    for attr in ('model', 'net', 'module', 'backbone'):
        sub = getattr(obj, attr, None)
        if sub is not None:
            n = _count_params(sub, depth+1, seen)
            if n > 0: return n
    return 0

def train_rfdetr(name, model_cls, cfg):
    print(f'\n{"="*60}\nTraining {name}  (resolution={RESOLUTION})\n{"="*60}')
    _free_gpu()
    out_dir = pathlib.Path('/content') / f'output_{name}'
    if out_dir.exists(): shutil.rmtree(out_dir)
    out_dir.mkdir()
    model = make_model(model_cls)
    n_params = _count_params(model)
    if n_params == 0:
        print(f'  WARNING: could not locate nn.Module on {model_cls.__name__}; reporting 0 params')
    t0 = time.time()
    model.train(output_dir=str(out_dir), **cfg)
    train_time = time.time() - t0
    rfdetr_results[name] = dict(
        params=n_params,
        train_time_s=train_time,
        best_weights=str(out_dir / 'checkpoint_best_total.pth'),
        output_dir=str(out_dir),
    )
    print(f'\n{name}: params={n_params/1e6:.2f}M  time={train_time/60:.1f}min')
    del model
    _free_gpu()

train_rfdetr('rfdetr_nano', RFDETRNano, TRAIN_CFG)

## Train RF-DETR Small

Included for within-architecture scaling, parallel to YOLO26's n/s/m/l. Adds about 1.5 hr on T4 / ~25-30 minutes on A100.

In [ ]:
train_rfdetr('rfdetr_small', RFDETRSmall, TRAIN_CFG)

## Evaluate on validation + test sets (mAP via supervision)

RF-DETR doesn't export the exact same metric names as Ultralytics, so we re-compute mAP50, mAP50-95, P, R via `supervision.metrics.MeanAveragePrecision` against the COCO ground truth, directly comparable to the YOLO numbers.

In [ ]:
import supervision as sv
from supervision.metrics import MeanAveragePrecision
from PIL import Image
from tqdm import tqdm
import numpy as np

def _iou_matrix(a, b):
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    union = aa[:, None] + bb[None, :] - inter
    return np.where(union > 0, inter / union, 0)

def _compute_pr(preds, targets, conf_thr=0.25, iou_thr=0.5):
    """Greedy IoU-matched precision/recall at a single threshold, comparable to YOLO defaults."""
    TP = FP = FN = 0
    for d, t in zip(preds, targets):
        keep = d.confidence > conf_thr
        det_boxes = d.xyxy[keep]; det_conf = d.confidence[keep]
        gt_boxes = t.xyxy
        if len(gt_boxes) == 0:
            FP += len(det_boxes); continue
        if len(det_boxes) == 0:
            FN += len(gt_boxes); continue
        iou = _iou_matrix(det_boxes, gt_boxes)
        matched_gt = set()
        for di in np.argsort(-det_conf):
            best_gt, best_iou = -1, iou_thr
            for gj in range(len(gt_boxes)):
                if gj in matched_gt: continue
                if iou[di, gj] >= best_iou:
                    best_iou, best_gt = iou[di, gj], gj
            if best_gt >= 0:
                matched_gt.add(best_gt); TP += 1
            else:
                FP += 1
        FN += len(gt_boxes) - len(matched_gt)
    P = TP / (TP + FP) if (TP + FP) > 0 else float('nan')
    R = TP / (TP + FN) if (TP + FN) > 0 else float('nan')
    return P, R, TP, FP, FN

def eval_rfdetr(name, model_cls, split):
    sp = pathlib.Path(DATASET_DIR) / split
    ds = sv.DetectionDataset.from_coco(
        images_directory_path=str(sp),
        annotations_path=str(sp / '_annotations.coco.json'),
    )
    model = make_model(model_cls, pretrain_weights=rfdetr_results[name]['best_weights'])  # same RESOLUTION as training
    model.optimize_for_inference()
    targets, preds = [], []
    for path, _img, anns in tqdm(ds, desc=f'{name}/{split}'):
        img = Image.open(path)
        d = model.predict(img, threshold=0.001)   # low threshold for AP curve coverage
        targets.append(anns); preds.append(d)
    metric = MeanAveragePrecision().update(preds, targets).compute()
    P, R, TP, FP, FN = _compute_pr(preds, targets, conf_thr=0.25, iou_thr=0.5)
    res = dict(
        mAP50     = float(metric.map50),
        mAP50_95  = float(metric.map50_95),
        precision = float(P),
        recall    = float(R),
        TP=int(TP), FP=int(FP), FN=int(FN),
    )
    del model; _free_gpu()
    return res

for _name, _cls in [('rfdetr_nano', RFDETRNano), ('rfdetr_small', RFDETRSmall)]:
    if _name not in rfdetr_results:
        continue  # skip variants that weren't trained
    rfdetr_results[_name]['val']  = eval_rfdetr(_name, _cls, 'valid')
    rfdetr_results[_name]['test'] = eval_rfdetr(_name, _cls, 'test')
    print(f'\n{_name}:')
    for split in ['val', 'test']:
        r = rfdetr_results[_name][split]
        print(f'  {split}: mAP50={r["mAP50"]:.4f}  mAP50-95={r["mAP50_95"]:.4f}  P={r["precision"]:.4f}  R={r["recall"]:.4f}  (TP={r["TP"]} FP={r["FP"]} FN={r["FN"]})')

## Comparison vs YOLO26

Pull YOLO26 numbers from the CSV produced by Step 2 and side-by-side.

In [ ]:
import pandas as pd

# YOLO26 reference numbers from Step 2
yolo_csv = '/content/drive/MyDrive/IE/IndividualAssignmentMBD2026/yolo26_comparison.csv'
try:
    df_yolo = pd.read_csv(yolo_csv, index_col=0)
    print('YOLO26 results (from Step 2):')
    print(df_yolo.to_string())
except FileNotFoundError:
    print(f'YOLO comparison CSV not found at {yolo_csv}; comparison table will only contain RF-DETR rows')
    df_yolo = pd.DataFrame()

# RF-DETR val + test tables (mirror Step 2's yolo26_comparison.csv + yolo26_test_metrics.csv)
val_rows, test_rows = [], []
for name, r in rfdetr_results.items():
    if 'val' not in r: continue
    val_rows.append({
        'model': name,
        'mAP@50': r['val']['mAP50'],
        'mAP@50-95': r['val']['mAP50_95'],
        'Precision': r['val']['precision'],
        'Recall': r['val']['recall'],
        'Params (M)': r['params']/1e6,
        'Train time (s)': r['train_time_s'],
    })
    if 'test' in r:
        test_rows.append({
            'model': name,
            'mAP@50': r['test']['mAP50'],
            'mAP@50-95': r['test']['mAP50_95'],
            'Precision': r['test']['precision'],
            'Recall': r['test']['recall'],
            'TP': r['test']['TP'], 'FP': r['test']['FP'], 'FN': r['test']['FN'],
        })

df_rfdetr_val  = pd.DataFrame(val_rows).set_index('model').round(4)
df_rfdetr_test = pd.DataFrame(test_rows).set_index('model').round(4)
print('\nRF-DETR val:')
print(df_rfdetr_val.to_string())
print('\nRF-DETR test:')
print(df_rfdetr_test.to_string())

df_rfdetr_val.to_csv('/content/rfdetr_comparison.csv')
df_rfdetr_test.to_csv('/content/rfdetr_test_metrics.csv')

## Failure case analysis (test set, best RF-DETR variant)

Mirrors Step 2's `yolo26_failures.zip`. For each test image we run the best RF-DETR variant (highest val mAP50-95) at conf=0.25 and classify it as TP-only / FP-only / FN-only / mixed by IoU match against GT. Annotated images are saved (green = TP, red = FP, yellow = missed GT) and zipped as `rfdetr_failures.zip` for direct side-by-side comparison with the YOLO failures in the write-up's RF-DETR vs YOLO section.

In [ ]:
import zipfile, cv2

# Pick the variant with highest val mAP50-95 for failure analysis
best_name = max(
    (n for n in rfdetr_results if 'val' in rfdetr_results[n]),
    key=lambda n: rfdetr_results[n]['val']['mAP50_95'],
)
best_cls = {'rfdetr_nano': RFDETRNano, 'rfdetr_small': RFDETRSmall}[best_name]
print(f'Saving failure cases for {best_name} (best val mAP50-95)')

model = make_model(best_cls, pretrain_weights=rfdetr_results[best_name]['best_weights'])  # same RESOLUTION as training
model.optimize_for_inference()

test_sp = pathlib.Path(DATASET_DIR) / 'test'
test_ds = sv.DetectionDataset.from_coco(
    images_directory_path=str(test_sp),
    annotations_path=str(test_sp / '_annotations.coco.json'),
)

failures_dir = pathlib.Path('/content/rfdetr_failures')
if failures_dir.exists(): shutil.rmtree(failures_dir)
failures_dir.mkdir()

CONF_THR, IOU_THR = 0.25, 0.5
counts = {'tp_only_or_clean': 0, 'fp_only': 0, 'fn_only': 0, 'mixed': 0}

for path, _img, gt in tqdm(test_ds, desc='failures'):
    img = Image.open(path).convert('RGB')
    det = model.predict(img, threshold=CONF_THR)
    det_boxes = det.xyxy; gt_boxes = gt.xyxy

    matched_gt, tp_idx, fp_idx = set(), [], []
    if len(det_boxes) and len(gt_boxes):
        iou = _iou_matrix(det_boxes, gt_boxes)
        for di in np.argsort(-det.confidence):
            best_gt, best_iou = -1, IOU_THR
            for gj in range(len(gt_boxes)):
                if gj in matched_gt: continue
                if iou[di, gj] >= best_iou:
                    best_iou, best_gt = iou[di, gj], gj
            if best_gt >= 0:
                matched_gt.add(best_gt); tp_idx.append(di)
            else:
                fp_idx.append(di)
    else:
        fp_idx = list(range(len(det_boxes)))
    fn_idx = [gj for gj in range(len(gt_boxes)) if gj not in matched_gt]

    n_fp, n_fn = len(fp_idx), len(fn_idx)
    if n_fp == 0 and n_fn == 0:
        counts['tp_only_or_clean'] += 1
        continue
    category = 'mixed' if (n_fp > 0 and n_fn > 0) else ('fp_only' if n_fp > 0 else 'fn_only')
    counts[category] += 1

    # Color tuples are in RGB order (NOT cv2's usual BGR), because `arr` came from PIL -> RGB
    # and we save via PIL -> RGB. cv2 draw funcs just write the tuple as raw channel bytes.
    arr = np.array(img)
    for i in tp_idx:
        x1,y1,x2,y2 = det_boxes[i].astype(int)
        cv2.rectangle(arr, (x1,y1), (x2,y2), (0,255,0), 2)       # green
        cv2.putText(arr, f'TP {det.confidence[i]:.2f}', (x1, max(y1-5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0,255,0), 1)
    for i in fp_idx:
        x1,y1,x2,y2 = det_boxes[i].astype(int)
        cv2.rectangle(arr, (x1,y1), (x2,y2), (255,0,0), 2)       # red
        cv2.putText(arr, f'FP {det.confidence[i]:.2f}', (x1, max(y1-5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,0,0), 1)
    for j in fn_idx:
        x1,y1,x2,y2 = gt_boxes[j].astype(int)
        cv2.rectangle(arr, (x1,y1), (x2,y2), (255,255,0), 2)     # yellow (R+G)
        cv2.putText(arr, 'FN', (x1, max(y1-5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,0), 1)

    Image.fromarray(arr).save(failures_dir / f'{category}_{pathlib.Path(path).stem}.png')

print(f'\nFailure breakdown for {best_name} on test:')
for cat, n in counts.items():
    print(f'  {cat}: {n}')

zip_path = '/content/rfdetr_failures.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in failures_dir.iterdir():
        zf.write(f, f.name)
print(f'\nSaved {zip_path} ({sum(1 for _ in failures_dir.iterdir())} annotated images)')

rfdetr_results['_failures'] = dict(variant=best_name, counts=counts, zip=zip_path)

del model; _free_gpu()

## Test predictions visualised

In [ ]:
import random, matplotlib.pyplot as plt

# Visualise predictions from the best variant (highest val mAP50-95)
best_name = max(
    (n for n in rfdetr_results if n != '_failures' and 'val' in rfdetr_results.get(n, {})),
    key=lambda n: rfdetr_results[n]['val']['mAP50_95'],
)
best_cls = {'rfdetr_nano': RFDETRNano, 'rfdetr_small': RFDETRSmall}[best_name]
print(f'Visualising {best_name}')
model = make_model(best_cls, pretrain_weights=rfdetr_results[best_name]['best_weights'])  # same RESOLUTION as training
model.optimize_for_inference()

test_dir = pathlib.Path(DATASET_DIR) / 'test'
test_imgs = [p for p in test_dir.iterdir() if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}]
random.seed(0)
sample = random.sample(test_imgs, k=min(9, len(test_imgs)))
box_ann = sv.BoxAnnotator(); label_ann = sv.LabelAnnotator(text_scale=0.5)
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for ax, p in zip(axes.flat, sample):
    img = Image.open(p)
    det = model.predict(img, threshold=0.5)
    labels = [f'pool {c:.2f}' for c in det.confidence]
    out = box_ann.annotate(scene=img.copy(), detections=det)
    out = label_ann.annotate(scene=out, detections=det, labels=labels)
    ax.imshow(out); ax.set_title(p.name, fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()
del model; _free_gpu()

## Save outputs to Drive

In [ ]:
drive_root = '/content/drive/MyDrive/IE/IndividualAssignmentMBD2026/'

# Per-model checkpoint zips
for name, r in rfdetr_results.items():
    if not isinstance(r, dict) or 'output_dir' not in r: continue
    shutil.make_archive(f'/content/{name}_output', 'zip', r['output_dir'])
    shutil.copy(f'/content/{name}_output.zip', drive_root)

# CSVs and failures zip
for artifact in ('/content/rfdetr_comparison.csv',
                 '/content/rfdetr_test_metrics.csv',
                 '/content/rfdetr_failures.zip'):
    if pathlib.Path(artifact).exists():
        shutil.copy(artifact, drive_root)
        print(f'  uploaded {pathlib.Path(artifact).name}')
print('\nDrive uploads complete')